In [0]:
dbutils.widgets.removeAll()

In [0]:
# DBTITLE 2, Environment and Dependency Setup
import os
import sys
from pathlib import Path

# Force workspace file system sync
os.sync()

# Set absolute workspace directory boundaries
ROOT_DIR = Path("/Workspace/Repos/logi@openhealthagents.org/claimspan/ClaimsProcessing")
sys.path.append(str(ROOT_DIR))

In [0]:
# DBTITLE 1, Define Production Pipeline Widgets
dbutils.widgets.text(
    "GoldConfigPath", 
    "/Workspace/Repos/logi@openhealthagents.org/claimspan/ClaimsProcessing/DimQualityYearMonth/Gold/Config/dimQualityYearMonth.json", 
    "Target Gold Config JSON Path"
)

# Extract config path directly from the widget
config_path = dbutils.widgets.get("GoldConfigPath")

In [0]:
# DBTITLE 2, Core Gold Layer Execution Function
def trigger_gold_processing(config_file_path: str):
    """Triggers the generalized Gold engine loop by passing the config file path."""
    # Define the exact path to your generic metadata notebook runner
    gold_notebook_path = f"{ROOT_DIR}/DimQualityYearMonth/Gold/Notebooks/GenericSubGroupProcessing"
    
    print(f"--> Invoking Gold Layer Engine...")
    print(f"    Target Config: {config_file_path}")
    
    payload_arguments = {
        "SubGroupConfigPath": config_file_path
    }
    
    # Run the generic processing engine
    result = dbutils.notebook.run(gold_notebook_path, 1200, arguments=payload_arguments)
    print(f"    Gold Layer Response: {result}")
    return result

In [0]:
# DBTITLE 3, Main Pipeline Sequenced Execution
def main():
    print("=======================================================")
    print("STARTING LIFECYCLE FOR PIPELINE: QualityYearMonth_Gold")
    print("=======================================================")
    
    try:
        # Run downstream Gold parsing table conversions and merge modifications
        pipeline_result = trigger_gold_processing(config_path)
        
        print("\n=======================================================")
        print("PIPELINE PROCESS COMPLETE")
        print(f"Final Outcome: {pipeline_result}")
        print("=======================================================")
        
        dbutils.notebook.exit(f"SUCCESS: {pipeline_result}")
        
    except Exception as e:
        error_msg = f"Pipeline execution faulted during processing: {str(e)}"
        print(f"CRITICAL: {error_msg}")
        dbutils.notebook.exit(f"FAILED: {error_msg}")

In [0]:
if __name__ == "__main__":
    main()

In [0]:
%sql
SELECT * FROM claimspan.gold.gold_dimqualityyearmonth;